In [22]:
import urllib.request
import ssl
import zipfile
import os
from pathlib import Path

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"

def download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path):
    if data_file_path.exists():
        print(f"{data_file_path} already exists. Skipping download and extraction.")
        return

    # Create an unverified SSL context
    ssl_context = ssl._create_unverified_context()

    # Downloading the file
    with urllib.request.urlopen(url, context=ssl_context) as response:
        with open(zip_path, "wb") as out_file:
            out_file.write(response.read())

    # Unzipping the file
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extracted_path)

    # Add .tsv file extension
    original_file_path = Path(extracted_path) / "SMSSpamCollection"
    os.rename(original_file_path, data_file_path)
    print(f"File downloaded and saved as {data_file_path}")

download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)


sms_spam_collection\SMSSpamCollection.tsv already exists. Skipping download and extraction.


In [23]:
import pandas as pd

df = pd.read_csv(data_file_path, sep= "\t", header= None, names= ['Label', 'Text'])
df

,Label,Text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [24]:
print(df['Label'].value_counts())

Label
ham     4825
spam     747
Name: count, dtype: int64


### **Stage 1: Preprocessing the dataset**


In [25]:
def create_balanced_dataset(df):
    
    # Count the instances of "spam"
    num_spam = df[df["Label"] == "spam"].shape[0]
    
    # Randomly sample "ham" instances to match the number of "spam" instances
    ham_subset = df[df["Label"] == "ham"].sample(num_spam, random_state=123)
    
    # Combine ham "subset" with "spam"
    balanced_df = pd.concat([ham_subset, df[df["Label"] == "spam"]])

    return balanced_df

balanced_df = create_balanced_dataset(df)
print(balanced_df["Label"].value_counts())

Label
ham     747
spam    747
Name: count, dtype: int64


In [26]:
balanced_df['Label'] = balanced_df['Label'].map({"ham" : 0, "spam" : 1})

In [27]:
balanced_df

,Label,Text
4307,0,Awww dat is sweet! We can think of something t...
4138,0,Just got to &lt;#&gt;
4831,0,"The word ""Checkmate"" in chess comes from the P..."
4461,0,This is wishing you a great day. Moji told me ...
5440,0,Thank you. do you generally date the brothas?
...,...,...
5537,1,Want explicit SEX in 30 secs? Ring 02073162414...
5540,1,ASKED 3MOBILE IF 0870 CHATLINES INCLU IN FREE ...
5547,1,Had your contract mobile 11 Mnths? Latest Moto...
5566,1,REMINDER FROM O2: To get 2.50 pounds free call...


In [28]:
def random_split(df, train_frac, validation_frac):
    # Shuffling the dateset
    df = df.sample(frac= 1, random_state= 42).reset_index(drop = True)

    # Calculating the split indices
    train_end = int(len(df) * train_frac)
    validatioin_end = train_end + int(len(df) * validation_frac)
    
    # Splitting the dataframes
    train_df = df[:train_end]
    validation_df = df[train_end : validatioin_end]
    test_df = df[validatioin_end : ]
    return train_df, validation_df, test_df

train_df, validation_df, test_df = random_split(balanced_df, 0.7, 0.1)

In [29]:
print(len(train_df))
print(len(validation_df))
print(len(test_df))

1045
149
300


In [30]:
train_df.to_csv('train.csv', index= False)
validation_df.to_csv('validation.csv', index= False)
test_df.to_csv('test.csv', index= False)

### **Stage 2: Implementing the dataloaders**


In [31]:
import re
from collections import Counter

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# Set a fixed seed so train/validation results are reproducible.
torch.manual_seed(123)

# Select GPU when it is available; otherwise keep the notebook runnable on CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cpu')

In [32]:
def simple_tokenizer(text):
    # Convert text to lowercase and split it into words and punctuation tokens.
    return re.findall(r"[A-Za-z0-9]+|[^\sA-Za-z0-9]", str(text).lower())


def build_vocab(texts, min_freq=1):
    # Count tokens only on the training split to avoid validation/test leakage.
    counter = Counter()
    for text in texts:
        counter.update(simple_tokenizer(text))

    # Reserve index 0 for padding and index 1 for unknown tokens.
    vocab = {"<pad>": 0, "<unk>": 1}
    for token, count in counter.most_common():
        if count >= min_freq and token not in vocab:
            vocab[token] = len(vocab)
    return vocab


def encode_text(text, vocab, max_length):
    # Map each token to its integer id and truncate long messages.
    token_ids = [vocab.get(token, vocab["<unk>"]) for token in simple_tokenizer(text)]
    token_ids = token_ids[:max_length]

    # Pad shorter messages so every batch can be stacked into one tensor.
    padding_length = max_length - len(token_ids)
    if padding_length > 0:
        token_ids = token_ids + [vocab["<pad>"]] * padding_length
    return token_ids


class SmsSpamDataset(Dataset):
    def __init__(self, csv_file, vocab=None, max_length=None):
        # Load one split from disk and keep the labels as integers for PyTorch.
        self.data = pd.read_csv(csv_file)
        self.labels = torch.tensor(self.data["Label"].values, dtype=torch.long)

        # Build the vocabulary from the training data, then reuse it for other splits.
        self.vocab = vocab if vocab is not None else build_vocab(self.data["Text"])

        # Use a robust message length from the training split unless one is provided.
        if max_length is None:
            lengths = [len(simple_tokenizer(text)) for text in self.data["Text"]]
            self.max_length = max(1, int(pd.Series(lengths).quantile(0.95)))
        else:
            self.max_length = max_length

        # Pre-encode all messages once so training batches are fast and consistent.
        encoded_messages = [encode_text(text, self.vocab, self.max_length) for text in self.data["Text"]]
        self.encoded_texts = torch.tensor(encoded_messages, dtype=torch.long)

    def __len__(self):
        # Return the number of SMS examples in this split.
        return len(self.labels)

    def __getitem__(self, index):
        # Return one encoded message and its matching label.
        return self.encoded_texts[index], self.labels[index]


train_dataset = SmsSpamDataset("train.csv")
validation_dataset = SmsSpamDataset("validation.csv", vocab=train_dataset.vocab, max_length=train_dataset.max_length)
test_dataset = SmsSpamDataset("test.csv", vocab=train_dataset.vocab, max_length=train_dataset.max_length)

print(f"Vocabulary size: {len(train_dataset.vocab)}")
print(f"Maximum sequence length: {train_dataset.max_length}")


Vocabulary size: 3757
Maximum sequence length: 41


In [33]:
batch_size = 32

# Shuffle only the training loader so the model sees examples in a different order each epoch.
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Inspect one batch to confirm the dataloader returns the expected tensor shapes.
features, labels = next(iter(train_loader))
print("Feature batch shape:", features.shape)
print("Label batch shape:", labels.shape)


Feature batch shape: torch.Size([32, 41])
Label batch shape: torch.Size([32])


### **Stage 3: Building the classification model**


In [34]:
class SmsSpamClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes=2, pad_idx=0):
        super().__init__()
        # Convert token ids into dense vectors; padding rows stay fixed at zeros.
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)

        # Classify the pooled message embedding with a compact feed-forward network.
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(hidden_dim, num_classes),
        )
        self.pad_idx = pad_idx

    def forward(self, input_ids):
        # Build a mask so padding tokens do not affect the average message embedding.
        mask = (input_ids != self.pad_idx).unsqueeze(-1)
        embeddings = self.embedding(input_ids)
        masked_embeddings = embeddings * mask

        # Average only real tokens and clamp the denominator to avoid division by zero.
        token_counts = mask.sum(dim=1).clamp(min=1)
        pooled_embeddings = masked_embeddings.sum(dim=1) / token_counts

        # Return raw class scores; CrossEntropyLoss applies softmax internally.
        return self.classifier(pooled_embeddings)


model = SmsSpamClassifier(
    vocab_size=len(train_dataset.vocab),
    embed_dim=64,
    hidden_dim=64,
).to(device)

model


SmsSpamClassifier(
  (embedding): Embedding(3757, 64, padding_idx=0)
  (classifier): Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=2, bias=True)
  )
)

### **Stage 4: Training and validation helpers**


In [35]:
def compute_accuracy(logits, labels):
    # Pick the class with the highest logit and compare it to the true labels.
    predictions = torch.argmax(logits, dim=1)
    return (predictions == labels).float().mean().item()


def evaluate_model(model, data_loader, loss_fn):
    # Switch to evaluation mode so dropout is disabled during scoring.
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    with torch.no_grad():
        for input_ids, labels in data_loader:
            # Move each batch to the same device as the model.
            input_ids = input_ids.to(device)
            labels = labels.to(device)

            logits = model(input_ids)
            loss = loss_fn(logits, labels)

            # Accumulate weighted totals so the final average handles short last batches.
            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_correct += (torch.argmax(logits, dim=1) == labels).sum().item()
            total_examples += batch_size

    return total_loss / total_examples, total_correct / total_examples


def train_model(model, train_loader, validation_loader, epochs=10, learning_rate=1e-3):
    # CrossEntropyLoss is appropriate for two-class classification with integer labels.
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history = []

    for epoch in range(1, epochs + 1):
        # Switch to training mode so dropout is active while fitting the model.
        model.train()
        running_loss = 0.0
        running_correct = 0
        running_examples = 0

        for input_ids, labels in train_loader:
            input_ids = input_ids.to(device)
            labels = labels.to(device)

            # Reset gradients, run a forward pass, and update the model weights.
            optimizer.zero_grad()
            logits = model(input_ids)
            loss = loss_fn(logits, labels)
            loss.backward()
            optimizer.step()

            # Track training metrics for this epoch.
            batch_size = labels.size(0)
            running_loss += loss.item() * batch_size
            running_correct += (torch.argmax(logits, dim=1) == labels).sum().item()
            running_examples += batch_size

        train_loss = running_loss / running_examples
        train_accuracy = running_correct / running_examples
        validation_loss, validation_accuracy = evaluate_model(model, validation_loader, loss_fn)

        # Store metrics so they can be reviewed after training finishes.
        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "validation_loss": validation_loss,
            "validation_accuracy": validation_accuracy,
        })

        print(
            f"Epoch {epoch:02d} | "
            f"train loss {train_loss:.4f}, train acc {train_accuracy:.3f} | "
            f"val loss {validation_loss:.4f}, val acc {validation_accuracy:.3f}"
        )

    return pd.DataFrame(history)


### **Stage 5: Training the spam classifier**


In [36]:
history_df = train_model(
    model,
    train_loader,
    validation_loader,
    epochs=10,
    learning_rate=1e-3,
)

# Display the training history as a table for quick inspection.
history_df


Epoch 01 | train loss 0.6451, train acc 0.601 | val loss 0.6100, val acc 0.698
Epoch 02 | train loss 0.5288, train acc 0.812 | val loss 0.4815, val acc 0.826
Epoch 03 | train loss 0.3698, train acc 0.895 | val loss 0.3534, val acc 0.846
Epoch 04 | train loss 0.2418, train acc 0.931 | val loss 0.2804, val acc 0.893
Epoch 05 | train loss 0.1780, train acc 0.951 | val loss 0.2430, val acc 0.899
Epoch 06 | train loss 0.1339, train acc 0.962 | val loss 0.2062, val acc 0.913
Epoch 07 | train loss 0.1011, train acc 0.974 | val loss 0.1895, val acc 0.919
Epoch 08 | train loss 0.0821, train acc 0.985 | val loss 0.1723, val acc 0.933
Epoch 09 | train loss 0.0670, train acc 0.989 | val loss 0.1618, val acc 0.933
Epoch 10 | train loss 0.0543, train acc 0.991 | val loss 0.1575, val acc 0.946


,epoch,train_loss,train_accuracy,validation_loss,validation_accuracy
0,1,0.645092,0.600957,0.610004,0.697987
1,2,0.528774,0.812440,0.481511,0.825503
2,3,0.369755,0.894737,0.353378,0.845638
3,4,0.241765,0.931100,0.280415,0.892617
4,5,0.177973,0.951196,0.243006,0.899329
5,6,0.133856,0.961722,0.206187,0.912752
6,7,0.101133,0.974163,0.189450,0.919463
7,8,0.082111,0.984689,0.172335,0.932886
8,9,0.066977,0.988517,0.161849,0.932886
9,10,0.054259,0.991388,0.157505,0.946309


### **Stage 6: Evaluating the final model**


In [37]:
loss_fn = nn.CrossEntropyLoss()

# Evaluate on data the model did not train on to estimate real-world performance.
test_loss, test_accuracy = evaluate_model(model, test_loader, loss_fn)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.3f}")


Test loss: 0.1840
Test accuracy: 0.930


In [38]:
def show_misclassified_examples(model, dataset, max_examples=5):
    # Review a few mistakes to understand what the classifier still finds difficult.
    model.eval()
    mistakes = []

    with torch.no_grad():
        for index in range(len(dataset)):
            input_ids, label = dataset[index]
            logits = model(input_ids.unsqueeze(0).to(device))
            prediction = torch.argmax(logits, dim=1).item()

            if prediction != label.item():
                mistakes.append({
                    "text": dataset.data.iloc[index]["Text"],
                    "actual_label": "spam" if label.item() == 1 else "ham",
                    "predicted_label": "spam" if prediction == 1 else "ham",
                })

            if len(mistakes) == max_examples:
                break

    return pd.DataFrame(mistakes)


show_misclassified_examples(model, test_dataset)


,text,actual_label,predicted_label
0,"4 tacos + 1 rajas burrito, right?",ham,spam
1,Great! I have to run now so ttyl!,ham,spam
2,"Velly good, yes please!",ham,spam
3,It vl bcum more difficult..,ham,spam
4,Want to finally have lunch today?,ham,spam


### **Stage 7: Making predictions on new messages**


In [39]:
def predict_sms(text, model, dataset):
    # Encode one raw SMS using the same vocabulary and sequence length used for training.
    model.eval()
    encoded_text = encode_text(text, dataset.vocab, dataset.max_length)
    input_ids = torch.tensor([encoded_text], dtype=torch.long).to(device)

    with torch.no_grad():
        logits = model(input_ids)
        probabilities = torch.softmax(logits, dim=1).squeeze(0)
        predicted_label = torch.argmax(probabilities).item()

    # Return both the class label and the spam probability for interpretability.
    return {
        "label": "spam" if predicted_label == 1 else "ham",
        "spam_probability": probabilities[1].item(),
    }


sample_messages = [
    "Congratulations! You won a free prize. Call now to claim.",
    "Can we meet at the library after lunch?",
]

for message in sample_messages:
    print(message)
    print(predict_sms(message, model, train_dataset))


Congratulations! You won a free prize. Call now to claim.
{'label': 'spam', 'spam_probability': 0.9999544620513916}
Can we meet at the library after lunch?
{'label': 'ham', 'spam_probability': 0.00044432844151742756}
